# Nuts & Bolts v3 — YOLO11n @ 320 (2 classes: bolt, nut)
Runtime → T4 GPU. Upload `nutsbolts_dataset.zip` (it contains ds3/). Built to hold 15 FPS on a Raspberry Pi 5.

In [ ]:
!pip -q install ultralytics
from google.colab import files
files.upload()
!unzip -q -o nutsbolts_dataset.zip -d /content/
print(open('/content/ds3/data.yaml').read())

In [ ]:
from ultralytics import YOLO
m = YOLO('yolo11n.pt')
m.train(data='/content/ds3/data.yaml', imgsz=320, epochs=200, batch=32, patience=30,
        # colour robustness: brown / black / silver / rusty parts, different lamps
        hsv_h=0.05, hsv_s=0.8, hsv_v=0.5,
        # orientation + scale: top-down parts lie at any angle and distance
        degrees=180, flipud=0.5, fliplr=0.5, scale=0.6, translate=0.15, perspective=0.0005,
        mosaic=1.0, mixup=0.1, close_mosaic=15, erasing=0.2,
        project='/content/runs', name='nb3')

In [ ]:
best = YOLO('/content/runs/nb3/weights/best.pt')
print(best.val(data='/content/ds3/data.yaml', imgsz=320))
best.export(format='onnx', imgsz=320)
files.download('/content/runs/nb3/weights/best.pt')
files.download('/content/runs/nb3/weights/best.onnx')